In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Pair Distribution Function: Si, NPD

This example demonstrates a pair distribution function (PDF) analysis
of Si, based on data collected from a time-of-flight neutron powder
diffraction experiment at NOMAD at SNS.

## 🛠️ Import Library

In [ ]:
import easydiffraction as ed

## 📦 Define Project

### Create Project

In [ ]:
project = ed.Project(name='si_nomad_pdf')

### Set Plotting Engine

In [ ]:
project.rendering_plot.show_supported()

In [ ]:
# Set global plot range for plots
project.rendering_plot.plotter.x_max = 40

### Add Structure

In [ ]:
project.structures.create(name='si')

In [ ]:
structure = project.structures['si']
structure.space_group.name_h_m.value = 'F d -3 m'
structure.space_group.coord_system_code = '1'
structure.cell.length_a = 5.43146
structure.atom_sites.create(
    id='Si',
    type_symbol='Si',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    adp_iso=0.5,
)

### Display Structure

In [ ]:
project.display.structure(struct_name='si')

### Add Experiment

In [ ]:
data_path = ed.download_data('meas-si-pdf-nomad', destination='data')

In [ ]:
project.experiments.add_from_data_path(
    name='nomad',
    data_path=data_path,
    sample_form='powder',
    beam_mode='time-of-flight',
    radiation_probe='neutron',
    scattering_type='total',
)

In [ ]:
experiment = project.experiments['nomad']
experiment.linked_structures.create(structure_id='si', scale=1.0)
experiment.peak.damp_q = 0.02
experiment.peak.broad_q = 0.03
experiment.peak.cutoff_q = 35.0
experiment.peak.sharp_delta_1 = 0.0
experiment.peak.sharp_delta_2 = 4.0
experiment.peak.damp_particle_diameter = 0

## 🚀 Perform Analysis

### Set Free Parameters

In [ ]:
project.structures['si'].cell.length_a.free = True
project.structures['si'].atom_sites['Si'].adp_iso.free = True
experiment.linked_structures['si'].scale.free = True

In [ ]:
experiment.peak.damp_q.free = True
experiment.peak.broad_q.free = True
experiment.peak.sharp_delta_1.free = True
experiment.peak.sharp_delta_2.free = True

### Run Fitting

In [ ]:
project.analysis.fit()
project.display.fit.results()
project.display.fit.correlations()

### Display Pattern

In [ ]:
project.display.pattern(expt_name='nomad')

## 💾 Save Project

In [ ]:
project.save_as(dir_path='projects/pdf-si-nomad')